### Step 1: Feature Extraction & SVM Classifier Training
This cell traverses the dataset, collects MRI images, extracts deep features using **MobileNetV2**, and trains a linear **SVM** classifier to categorize tumor types.

In [ ]:
import os
import random
import csv
import numpy as np
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing import image
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib

# 1. Correct dataset traversal for Training & Testing folder structure
archive_dir = None
for root, dirs, files in os.walk('MediScan'):
    if 'Training' in dirs and 'Testing' in dirs:
        archive_dir = root
        break
if not archive_dir:
    archive_dir = 'MediScan/archive'

print(f"Dataset root directory: {archive_dir}")

train_sources = [os.path.join(archive_dir, 'Training')]
test_sources = [os.path.join(archive_dir, 'Testing')]

def collect_images(sources):
    data = []
    for src in sources:
        if not os.path.exists(src):
            continue
        for cls in sorted(os.listdir(src)):
            cls_path = os.path.join(src, cls)
            if os.path.isdir(cls_path):
                for root, _, files in os.walk(cls_path):
                    for file in files:
                        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
                            rel_path = os.path.relpath(os.path.join(root, file), archive_dir)
                            data.append((rel_path, cls))
    return data

train_data = collect_images(train_sources)
test_data = collect_images(test_sources)

os.makedirs('MediScan/mediScan_splits', exist_ok=True)
with open('MediScan/mediScan_splits/train.txt', 'w', newline='') as f:
    csv.writer(f, delimiter='\t').writerows(train_data)
with open('MediScan/mediScan_splits/test.txt', 'w', newline='') as f:
    csv.writer(f, delimiter='\t').writerows(test_data)

print(f"Corrected Splits! Train: {len(train_data)}, Test: {len(test_data)}")

# 2. Feature Extraction Function
def extract_features(manifest_path, output_dir):
    model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')
    paths, labels = [], []
    with open(manifest_path, 'r') as f:
        reader = csv.reader(f, delimiter='\t')
        for row in reader:
            paths.append(row[0])
            labels.append(row[1])

    features, valid_paths, valid_labels = [], [], []
    print(f"Extracting features for {len(paths)} images into {output_dir}...")

    for i, path in enumerate(paths):
        full_path = os.path.join(archive_dir, path)
        try:
            img = image.load_img(full_path, target_size=(224, 224))
            img_array = image.img_to_array(img)
            if img_array.shape[-1] != 3:
                img_array = np.stack((img_array,)*3, axis=-1)
            img_array = np.expand_dims(img_array, axis=0)
            processed_img = preprocess_input(img_array)

            feat = model.predict(processed_img, verbose=0)
            features.append(feat.flatten())
            valid_paths.append(path)
            valid_labels.append(labels[i])
        except Exception as e:
            pass

    os.makedirs(output_dir, exist_ok=True)
    np.save(os.path.join(output_dir, 'embeddings.npy'), np.array(features))
    np.save(os.path.join(output_dir, 'labels.npy'), np.array(valid_labels))
    print(f"Completed {output_dir}. Shape: {np.array(features).shape}")

extract_features('MediScan/mediScan_splits/train.txt', 'MediScan/features')
extract_features('MediScan/mediScan_splits/test.txt', 'MediScan/features_test')

# 3. Train SVM Classifier with Correct Tumor Classes
print("\nTraining SVM Classifier on Tumor Classes...")
X_train = np.load('MediScan/features/embeddings.npy')
y_train = np.load('MediScan/features/labels.npy')
X_test = np.load('MediScan/features_test/embeddings.npy')
y_test = np.load('MediScan/features_test/labels.npy')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm_model = SVC(kernel='linear', C=1.0, random_state=42)
svm_model.fit(X_train_scaled, y_train)

y_pred = svm_model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)

print(f"\n==============================")
print(f"Corrected SVM Test Accuracy: {acc * 100:.2f}%")
print(f"==============================")
print(classification_report(y_test, y_pred, zero_division=0))

os.makedirs('MediScan/models', exist_ok=True)
joblib.dump(svm_model, 'MediScan/models/svm_model.joblib')
joblib.dump(scaler, 'MediScan/models/scaler.joblib')
print("SVM model and scaler saved successfully!")

Dataset root directory: MediScan/archive
Corrected Splits! Train: 5600, Test: 1200


/tmp/ipykernel_1473/193755181.py:54: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')


Extracting features for 5600 images into MediScan/features...
Completed MediScan/features. Shape: (5600, 1280)
Extracting features for 1200 images into MediScan/features_test...
Completed MediScan/features_test. Shape: (1200, 1280)

Training SVM Classifier on Tumor Classes...

Corrected SVM Test Accuracy: 87.00%
              precision    recall  f1-score   support

      glioma       0.92      0.72      0.81       400
  meningioma       0.79      0.92      0.85       400
     notumor       0.00      0.00      0.00         0
   pituitary       0.97      0.97      0.97       400

    accuracy                           0.87      1200
   macro avg       0.67      0.65      0.66      1200
weighted avg       0.89      0.87      0.88      1200

SVM model and scaler saved successfully!


### Step 2: Image Retrieval Engine (CBIR Test)
This cell sets up the **Nearest Neighbors** (Cosine Similarity) model to find the most visually similar MRI scans from our trained library based on a given query image.

In [ ]:
import os
import csv
import numpy as np
from sklearn.neighbors import NearestNeighbors
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing import image

# 1. Generate missing image_paths.csv files for Train and Test
for split_name, folder_name in [('train.txt', 'MediScan/features'), ('test.txt', 'MediScan/features_test')]:
    manifest_path = os.path.join('MediScan/mediScan_splits', split_name)
    paths = []
    if os.path.exists(manifest_path):
        with open(manifest_path, 'r') as f:
            reader = csv.reader(f, delimiter='\t')
            for row in reader:
                paths.append([row[0]])

        os.makedirs(folder_name, exist_ok=True)
        with open(os.path.join(folder_name, 'image_paths.csv'), 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerows(paths)

print("image_paths.csv generated successfully! Running Retrieval Test...")

# 2. Retrieval Test Class & Execution
class MediScanRetrieval:
    def __init__(self):
        self.embeddings = np.load('MediScan/features/embeddings.npy')
        self.labels = np.load('MediScan/features/labels.npy')
        self.archive_dir = 'MediScan/archive'

        self.image_paths = []
        with open('MediScan/features/image_paths.csv', 'r') as f:
            for row in csv.reader(f):
                self.image_paths.append(row[0])

        self.nn_model = NearestNeighbors(n_neighbors=5, metric='cosine')
        self.nn_model.fit(self.embeddings)
        self.model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')

    def retrieve_similar(self, query_img_path):
        img = image.load_img(query_img_path, target_size=(224, 224))
        img_array = image.img_to_array(img)
        if img_array.shape[-1] != 3:
            img_array = np.stack((img_array,)*3, axis=-1)
        img_array = np.expand_dims(img_array, axis=0)
        processed_img = preprocess_input(img_array)

        query_feat = self.model.predict(processed_img, verbose=0).flatten().reshape(1, -1)
        distances, indices = self.nn_model.kneighbors(query_feat)

        results = []
        for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
            results.append({
                'rank': rank + 1,
                'path': self.image_paths[idx],
                'label': self.labels[idx],
                'similarity_score': float(1.0 - dist)
            })
        return results

retriever = MediScanRetrieval()

# Test query from test set
with open('MediScan/features_test/image_paths.csv', 'r') as f:
    first_test_img = next(csv.reader(f))[0]

full_query_path = os.path.join(retriever.archive_dir, first_test_img)
print(f"\nQuery Image Path: {first_test_img}")

top_matches = retriever.retrieve_similar(full_query_path)
print("\nTop 5 Similar Images Retrieved:")
for match in top_matches:
    print(f"  {match['rank']}. Path: {match['path']} | Label: {match['label']} | Similarity: {match['similarity_score']:.4f}")

image_paths.csv generated successfully! Running Retrieval Test...


/tmp/ipykernel_1473/2985995827.py:39: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  self.model = MobileNetV2(weights='imagenet', include_top=False, pooling='avg')



Query Image Path: Testing/glioma/Te-gl_311.jpg

Top 5 Similar Images Retrieved:
  1. Path: Training/glioma/Tr-gl_1057.jpg | Label: glioma | Similarity: 1.0000
  2. Path: Training/glioma/Tr-gl_1047.jpg | Label: glioma | Similarity: 0.9578
  3. Path: Training/glioma/Tr-gl_824.jpg | Label: glioma | Similarity: 0.8526
  4. Path: Training/glioma/Tr-gl_883.jpg | Label: glioma | Similarity: 0.8404
  5. Path: Training/glioma/Tr-gl_710.jpg | Label: glioma | Similarity: 0.8380
